### 라이브러리

In [69]:
import pandas as pd
import numpy as np

### 데이터 호출

In [70]:
df = pd.read_csv('train.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 908 entries, 0 to 907
Columns: 2881 entries, PRODUCT_ID to X_2875
dtypes: float64(2876), int64(1), str(4)
memory usage: 20.0 MB


### STEP1 누수 컬럼 제거 

In [71]:
def drop_leakage(df, 
                 target_col='Y_Quality',
                 id_cols = ['PRODUCT_CODE'],
                 time_cols = 'TIMESTAMP'):
    y = df[target_col].copy()
    temp = df[id_cols + [time_cols]].copy()
    X = df.drop(columns=[target_col] + id_cols + [time_cols])
    return X,y,temp

### STEP2 완전 결측 컬럼 제거

In [72]:
def remove_all_nan(X):
    all_nan_cols = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_nan_cols)
    return X, all_nan_cols

### STEP3 상수/저분산 제거

In [73]:
def remove_constant(X):
    nunique = X.columns[X.nunique(dropna=False) <= 1]
    std_zero = X.select_dtypes(include="number").columns[
        X.select_dtypes(include="number").std() == 0
    ]
    
    drop_cols = list(set(nunique) | set(std_zero))
    X = X.drop(columns=drop_cols)
    
    return X, drop_cols

### STEP4 위장 결측 -> NaN

In [74]:
def missing_value(X, placeholder = [999,9999,-999,-9999]):
    X = X.replace(placeholder, np.nan)
    return X

### STEP5 제품별 분리 

In [75]:
def split_by_product(df, product_col='PRODUCT_CODE'):
    df_A = df[df[product_col] == 'A_31'].copy()
    df_TO = df[df[product_col].isin(["T_31","O_31"])].copy()
 
    return df_A, df_TO 

### STEP6 라인 인코딩

In [76]:
def encode_line(X, line_col='LINE'):
    if line_col in X.columns:
        X = pd.get_dummies(X, columns=[line_col], drop_first=True)
    return X

### 전체 전처리 함수 

In [77]:
def preprocess(
    df,
    target_col='Y_Quality',
    id_cols=['PRODUCT_CODE'],
    time_cols='TIMESTAMP',
    placeholder=[999, 9999, -999, -9999],
    line_col='LINE',
    remove_nan=True
):
    # STEP1 누수 컬럼 제거
    X, y, temp = drop_leakage(
        df,
        target_col=target_col,
        id_cols=id_cols,
        time_cols=time_cols
    )

    # STEP2 완전 결측 컬럼 제거
    if remove_nan:
        X, all_nan_cols = remove_all_nan(X)
    else:
        all_nan_cols = X.columns[X.isna().all()].tolist()

    # STEP3 상수/저분산 제거
    X, constant_cols = remove_constant(X)

    # STEP4 위장결측 → NaN
    X = missing_value(X, placeholder)

    # STEP6 LINE 인코딩
    X = encode_line(X, line_col)

    print("=" * 50)
    print(f"최종 데이터 크기 : {X.shape}")
    print(f"완전결측 제거 : {len(all_nan_cols)}개")
    print(f"상수/저분산 제거 : {len(constant_cols)}개")
    print("=" * 50)

    return X, y, temp

### 전처리 적용

In [78]:
df_A, df_TO = split_by_product(df)
X_A, y_A, temp_A = preprocess(df_A)
X_TO, y_TO, temp_TO = preprocess(df_TO)

print(X_A.shape)

print(X_TO.shape)

최종 데이터 크기 : (316, 1968)
완전결측 제거 : 681개
상수/저분산 제거 : 231개
최종 데이터 크기 : (592, 571)
완전결측 제거 : 2194개
상수/저분산 제거 : 113개
(316, 1968)
(592, 571)


### CSV 형식으로 저장

In [79]:
df_A_processed = pd.concat([X_A, y_A, temp_A], axis=1)
df_A_processed.to_csv("A_31_preprocessed.csv", index=False, encoding="utf-8-sig")

df_TO_processed = pd.concat([X_TO, y_TO, temp_TO], axis=1)
df_TO_processed.to_csv("TO_31_preprocessed.csv", index=False, encoding="utf-8-sig")